# 10 · X3: the final retrieval pipeline (phase A: GPU encoder)

X3 = BM25 + dense → RRF (k = 10) → keyword engine-type filter → 30 candidates → jina-reranker-v3 → score cut-off ≥ 0.0, keep 1–8 (params `v3`, windows index `cse_win500_v1`). The notebook runs X3 on the questions of the active experiment, measures it like every other run, compares it with X2, and saves the retrieved contexts for the generation notebooks. Design and numbers: `x2_to_x3_plan.md`.

> **⚠ STOP vLLM BEFORE RUNNING THIS NOTEBOOK.** Qdrant and the **GPU encoder** (`encoder-gpu`) must be running, and the GPU encoder and vLLM do not fit on the card together.
> ```bash
> docker compose stop vllm encoder                   # vLLM off; the CPU encoder shares port 8081
> docker compose up -d qdrant && docker compose up -d --force-recreate encoder-gpu
> ```
> The guard cell below refuses to run while vLLM answers on port 8000, and refuses to run on a CPU encoder unless you set `ALLOW_CPU_RERANK = True`.
>
> **After this notebook**, the generation notebooks need only vLLM, because they read `results/<experiment>/x3/contexts.jsonl` and use neither Qdrant nor the encoder:
> ```bash
> docker compose stop encoder-gpu && docker compose up -d vllm      # VLLM_CONFIG=serve.retrieval.yaml (.env); ~2.5 min to healthy
> ```

| step | what |
|---|---|
| 1 | parameters, the keyword rules, the X3 config, the baselines (the stored v1 runs X2 gate off and on, H1) |
| 2 | the keyword rule against the labels; **the gold sections the rule would remove** |
| 3 | run X3 (resumable, cached) |
| 4 | the standard metric files of notebook 09 for X3 and the baselines, and the pipeline diagram |
| 5 | the X2 vs X3 table, by family, paired differences, the questions where X3 loses evidence |
| 6 | other cut-offs on X3's own reranked lists (no reranker call) |
| 7 | the unanswerable questions (inspection only) |
| 8 | the hand-off file `contexts.jsonl` |
| 9 | the checks |

The X3 **run record** goes to `results/<experiment>/runs/` next to the v1 runs. Every other file is in `results/<experiment>/x3/`: `contexts.jsonl`, `run_metrics.parquet`, `question_metrics.parquet`, `win500/metrics.{csv,xlsx}`, `comparison.csv`, `by_family.csv`, `paired.csv`, `evidence_losses.csv`, `cut_sweep.csv`, `keyword_rule.csv`, `diagrams/`.

Everything is cached and the run resumes from `partial.jsonl`. The first run makes the jina calls (all new: jina is cached per ordered candidate list); the filtered first-stage lists of the questions that the starting keyword lists detect are already in `data/cache/search.jsonl`. After the first run every step works from the caches: set `SERVICES = False` to run steps 2 and 4–9 with no Qdrant and no encoder (for example while vLLM is up). Editing `engine_keywords.toml` gives a new run id and a new set of jina calls; the old run stays.

X3's run record holds the windows the LLM would get (1–8), as the gate-on runs do, so its `@10` columns describe a short list. The columns to decide on are `section_hit@1`, `section_hit@3`, `evidence_recall@ctx` and the context tokens.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
from csrag import experiment
EXP = experiment.active()  # `active` in experiments.toml, or the CSRAG_EXPERIMENT environment variable
RESULTS = ROOT / EXP.dir   # results/<experiment name>/: every result of this experiment
print(EXP.summary())

In [ ]:
import json, tempfile, time
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from tqdm.auto import tqdm
from csrag import engine_rule, evaluate as ev, final, report as rp, runs
from csrag.corpus import COLLECTIONS, Corpus
from csrag.retrievers import Pipeline, Resources, grid, method_config, prompt_hashes
from csrag.retrievers.query_llm import vllm_healthy

PARAMS_VERSION = "v3"  # X3's parameters; the baselines are the stored v1 runs
INDEX = "win500"  # X3 runs on the windows index only
SERVICES = True  # False: no Qdrant and no encoder, everything from the caches (needs a completed X3 run)
ALLOW_CPU_RERANK = False  # True only if you accept hours of CPU reranking (use the GPU encoder)
ABLATIONS = []  # optional extra real runs, one more jina pass each: any of "no_filter", "rerank_input_50"
MAX_EXCLUDED_GOLD = 2  # check 6: questions whose gold section the keyword rule removes
TARGET_HIT1, TARGET_EVIDENCE = 0.90, 0.95  # check 2: overall Hit@1 and evidence in context
SWEEP_MIN_SCORES, SWEEP_MAX_KEEPS = [-0.05, 0.0, 0.05, 0.1], [5, 8, 10, 15]  # step 6
OUT = RESULTS / "x3"  # every file of the final pipeline except the run record, which is in runs/

assert set(ABLATIONS) <= set(final.ABLATION_LABELS), f"unknown ablation; defined: {sorted(final.ABLATION_LABELS)}"
corpus = Corpus(DATA)
assert corpus.has_windows, "run notebook 04 first"
if SERVICES:
    assert not vllm_healthy(), "vLLM is running: `docker compose stop vllm`, then `docker compose up -d qdrant && docker compose up -d --force-recreate encoder-gpu`"
    res = Resources.connect(corpus, DATA, with_llm=False)
    health = res.encoder.wait_ready()
    print("encoder on", health["device"], health.get("gpu") or "", "| dtype", health["dtype"])
    assert health["device"] == "cuda" or ALLOW_CPU_RERANK, (
        "the encoder runs on the CPU (hours of reranking): `docker compose stop encoder && docker compose up -d --force-recreate encoder-gpu`, "
        "or set ALLOW_CPU_RERANK = True")
    n = res.client.get_collection(COLLECTIONS[INDEX]).points_count
    assert n == len(corpus.units(INDEX)), f"{INDEX}: {n} points, run notebook 05"
    print("Qdrant collection", COLLECTIONS[INDEX], "ready")  # no ColBERT check: X3 does not use ColBERT
else:
    res = Resources.connect(corpus, DATA, with_llm=False, with_encoder=False)
    print("SERVICES = False: working from the caches only")

## 1. Questions, keyword rules, X3 config and baselines

The baselines are looked up by their config in this experiment's `runs/`: X2 (BM25 + dense → RRF k = 60 → 50 candidates → jina → top 20, gate off), X2 with gate B, and H1 (the hybrid before the reranker), all v1 on windows.

In [ ]:
questions, problems = ev.load_questions(EXP.questions, corpus)
assert not any(v for k, v in problems.items() if k != "quote_ambiguous"), "question validation failed: run notebook 06"
qrels = ev.read_qrels(EXP.qrels_path)
answerable = [q for q in questions if q["answerable"]]
assert {r["question_id"] for r in qrels} == {q["question_id"] for q in answerable}, "qrels do not match the questions: run notebook 06"
families = list(dict.fromkeys(ev.family_of(q) for q in answerable))
prompts = prompt_hashes()  # only for check 1 (the v1 grid)

rules = engine_rule.load()  # engine_keywords.toml, validated and normalised
rules_hash = engine_rule.rules_hash(rules)
x3_cfg = method_config("X3", INDEX, False, PARAMS_VERSION, engine_rules_hash=rules_hash)
X3_ID = runs.run_id_of(x3_cfg)
X2_ID = final.find_run("X2", INDEX, False, "v1")
X2_ON_ID = final.find_run("X2", INDEX, True, "v1")
H1_ID = final.find_run("H1", INDEX, False, "v1")
BASELINES = [X2_ID, X2_ON_ID, H1_ID]
stale = [r for r in BASELINES if not runs.same_questions(r, questions)]
assert not stale, f"baseline runs made with another question set: {stale}"

print(f"{len(questions)} questions, {len(answerable)} answerable | families: {families}")
print("X3 run id:", X3_ID, "| keyword rules:", rules_hash)
print("baselines:", BASELINES)
print(json.dumps({k: x3_cfg[k] for k in ("first_stages", "candidates", "fusion", "filter", "rerank", "cut", "k")}, indent=1))

## 2. The keyword rule against the labels

The rule reads the question text only. The labels (`engine_type` of each question) are used here to see how the rule performs, and for nothing else. A wrong filter removes the gold section completely: it is the only way X3 can do worse than X2 on a question, so the questions whose gold section the rule would remove are listed, and the notebook stops here if there are more than `MAX_EXCLUDED_GOLD`, before any reranker time is spent. Edit `engine_keywords.toml`, then re-run steps 1 and 2 (the rules are read in step 1).

In [ ]:
print(json.dumps(rules, indent=1, ensure_ascii=False))
kr = final.keyword_report(questions, rules, corpus, INDEX)
by_label, reasons = final.keyword_summary(kr)
display(Markdown("**Label (rows) against detected type (columns)**"))
display(by_label)
display(Markdown("**Detection reasons**"))
display(reasons.rename("questions").to_frame().T)
excluded = kr[kr["n_excluded_gold"] > 0]
display(Markdown(f"**{len(excluded)} question(s) whose gold section the rule would remove** (limit {MAX_EXCLUDED_GOLD})"))
display(excluded[["question_id", "family", "label", "detected", "matched", "excluded_gold", "question"]])
OUT.mkdir(parents=True, exist_ok=True)
kr.to_csv(OUT / "keyword_rule.csv", index=False, encoding="utf-8")
assert len(excluded) <= MAX_EXCLUDED_GOLD, (
    "the keyword rule removes too many gold sections: edit engine_keywords.toml and re-run steps 1 and 2 "
    "(or raise MAX_EXCLUDED_GOLD) before the run in step 3")

## 3. Run X3

`runs.execute` is resumable (`partial.jsonl`) and uses the caches; the jina calls are the new work. The record goes to `results/<experiment>/runs/`. The optional ablations (`ABLATIONS`) are separate run records.

In [ ]:
cfgs = {"X3": x3_cfg, **{a: final.ablation_config(a, INDEX, PARAMS_VERSION, rules_hash) for a in ABLATIONS}}
run_ids = {name: runs.run_id_of(cfg) for name, cfg in cfgs.items()}
for name, cfg in cfgs.items():
    rid = run_ids[name]
    assert SERVICES or runs.is_complete(rid), f"{rid} is not complete: run the notebook once with SERVICES = True"
    if SERVICES:
        assert not vllm_healthy(), "vLLM is running: `docker compose stop vllm` before reranking"
    t0 = time.time()
    runs.execute(cfg, Pipeline(cfg, res, engine_rules=rules).run, questions, res.meta(cfg["index"]), progress=tqdm)
    print(rid, "|", round(time.time() - t0), "s")
assert run_ids["X3"] == X3_ID

## 4. The standard metric files

The same files notebook 09 writes (`run_metrics.parquet`, `question_metrics.parquet`, `win500/metrics.{csv,xlsx}` with all sheets), for X3 and the three baselines, into `results/<experiment>/x3/`. The code is notebook 09's (`ev.evaluate_all`, `rp.write_index_files`), so every metric, CI and breakdown is computed as for every other run. The `significance` sheet stays empty: the paired comparisons are in step 5. Re-running notebook 09 afterwards would add X3 as one more row to the main tables with the same values.

In [ ]:
metric_ids = list(dict.fromkeys([X3_ID, *[run_ids[a] for a in ABLATIONS], *BASELINES]))
w = final.write_metric_files(metric_ids, questions, qrels, corpus, OUT, INDEX, progress=tqdm)
table_csv = pd.read_csv(OUT / INDEX / "metrics.csv", encoding="utf-8")
COLS = ["method", "gate", "params_version", "filter", "rerank", "section_hit@1", "section_hit@3", "section_ndcg@10", "section_recall@10",
        "evidence_recall@10", "evidence_recall@ctx", "context_tokens", "n_hits", "latency_rerank_s", "latency_total_s"]
display(table_csv.set_index("run_id")[[c for c in COLS if c in table_csv.columns]].round(3))
diagram = rp.write_diagram(X3_ID, OUT / "diagrams", rm=w["run_metrics"], sig=None, gates=w["gates"], costs=w["costs"])
display(Markdown(diagram.read_text(encoding="utf-8")))

## 5. X2 vs X3

| row | what it isolates |
|---|---|
| X2, 20 returned | the baseline as reported |
| X2, top 8 | a fixed cut of the same size limit (replay of the stored list) |
| X2 + X3's cut-off | the cut-off alone (replay of the stored list) |
| X2, gate on | the old way to shorten the context |
| X3, no cut-off (30 windows) | what the filter and the 30 candidates give before the cut (replay of `stats["reranked"]`); its nDCG@10 and Recall@10 can be set against X2's 20 |
| **X3** | everything |

Section level; 95% bootstrap CIs on the same resamples for every row. Reranker and total seconds are those of the run that made the jina call (a replay row shares the call of its source run). Context precision is the share of returned windows that lie in a gold section. Differences are paired by question; `p_holm` corrects across the six metrics within each pair. As noted in the README, pooled CIs and tests treat the questions as independent although a paraphrase shares its gold with its original, so they are too narrow: read the families separately.

In [ ]:
CUT = x3_cfg["cut"]
X2_LABEL, X2_ON_LABEL, X3_LABEL = "X2, 20 returned", "X2, gate on", "X3"
mo = lambda hits: final.metrics_of(hits, questions, qrels, corpus, INDEX)
rows = {
    X2_LABEL: mo(runs.load_hits(X2_ID)),
    "X2, top 8": mo(final.replay_cut(X2_ID, corpus, -np.inf, 8, 8)),
    "X2 + X3's cut-off": mo(final.replay_cut(X2_ID, corpus, **CUT)),
    X2_ON_LABEL: mo(runs.load_hits(X2_ON_ID)),
    "X3, no cut-off (30 windows)": mo(final.replay_cut(X3_ID, corpus, **final.UNCUT)),
    X3_LABEL: mo(runs.load_hits(X3_ID)),
    **{final.ABLATION_LABELS[a]: mo(runs.load_hits(run_ids[a])) for a in ABLATIONS},
}
source = {X2_LABEL: X2_ID, "X2, top 8": X2_ID, "X2 + X3's cut-off": X2_ID, X2_ON_LABEL: X2_ON_ID, "X3, no cut-off (30 windows)": X3_ID,
          X3_LABEL: X3_ID, **{final.ABLATION_LABELS[a]: run_ids[a] for a in ABLATIONS}}
lat = {label: final.latency(rid) for label, rid in source.items()}
table, by_family, paired = final.compare(rows, [(X3_LABEL, X2_LABEL), (X3_LABEL, "X2 + X3's cut-off")], lat)
SHOW = ["row", "n", "hit@1", "hit@1_ci_lo", "hit@1_ci_hi", "hit@3", "hit@3_ci_lo", "hit@3_ci_hi", "mrr@10", "ndcg@10", "recall@10", "evidence_ctx",
        "windows_mean", "windows_median", "tokens_mean", "precision", "rerank_s", "total_s"]
display(table[[c for c in SHOW if c in table.columns]].set_index("row").round(3))
table.to_csv(OUT / "comparison.csv", index=False, encoding="utf-8")
by_family.to_csv(OUT / "by_family.csv", index=False, encoding="utf-8")
paired.to_csv(OUT / "paired.csv", index=False, encoding="utf-8")

In [ ]:
for metric in ("hit@1", "evidence_ctx"):
    display(Markdown(f"**{metric} by family**"))
    display(by_family.pivot(index="row", columns="family", values=metric).reindex(index=list(rows), columns=families).round(3))
display(Markdown("**twin_topic: questions with a section of the other engine type at rank 1**"))
display(pd.DataFrame({X2_LABEL: final.other_type_at_rank1(runs.load_hits(X2_ID), questions, corpus, INDEX),
                      X3_LABEL: final.other_type_at_rank1(runs.load_hits(X3_ID), questions, corpus, INDEX)}).T)
display(Markdown("**Paired differences (a − b)**"))
display(paired.round(4))

In [ ]:
loss = final.evidence_losses(X3_ID, runs.load_hits(X2_ID), questions, qrels, corpus)
loss.to_csv(OUT / "evidence_losses.csv", index=False, encoding="utf-8")
display(Markdown(f"**{len(loss)} question(s) where X3's context has less evidence than X2's 20 windows**, by cause "
                 "(`filter`: the keyword rule removed a gold section; `candidates`: not among the 30 sent to jina; `cut`: dropped by the cut-off)"))
display(loss["cause"].value_counts().rename("questions").to_frame().T)
display(loss.drop(columns="question").round(3))

## 6. Other cut-offs on X3's own reranked lists

`stats["reranked"]` holds every candidate with its jina score, so other thresholds cost no reranker call. `cut_loss` is the share of questions whose evidence in context is lower than that of the whole uncut ordering. The second table is the configured cut-off by family.

In [ ]:
sweep = final.cut_sweep(X3_ID, questions, qrels, corpus, SWEEP_MIN_SCORES, SWEEP_MAX_KEEPS, min_keep=CUT["min_keep"])
sweep.to_csv(OUT / "cut_sweep.csv", index=False, encoding="utf-8")
display(sweep[sweep["family"] == "all"].drop(columns="family").round(3))
display(Markdown(f"**The configured cut-off (score ≥ {CUT['min_score']}, keep {CUT['min_keep']}–{CUT['max_keep']}) by family**"))
cols = ["family", "n", "hit@1", "hit@3", "evidence_ctx", "windows_mean", "tokens_mean", "cut_loss"]
display(sweep[(sweep["min_score"] == CUT["min_score"]) & (sweep["max_keep"] == CUT["max_keep"])][cols].round(3))

## 7. Unanswerable questions (inspection only, not scored)

X3 always returns at least one window. The scores show how a reader (or the LLM) could tell that nothing answers the question.

In [ ]:
un = final.unanswerable_view(X3_ID, questions, corpus)
display(un.round(3) if len(un) else "no unanswerable questions in this experiment")

## 8. The hand-off file

`results/<experiment>/x3/contexts.jsonl`, one line per question. `hits` is what X3 hands over (1–8 windows with their text) and goes into the prompt; `reranked` is every candidate with its jina score, best first, without text (the text of a window is `corpus.units("win500")[unit_id]["text"]`). A generation notebook calls `final.read_contexts(path, question_ids=[...])` with 3–5 ids and needs only vLLM. Gold answers stay in the question files and are joined by `question_id`.

In [ ]:
contexts_path = final.write_contexts(X3_ID, questions, corpus, OUT / "contexts.jsonl")
print(contexts_path, "|", round(contexts_path.stat().st_size / 1e6, 1), "MB |", len(final.read_contexts(contexts_path)), "questions")
for r in final.read_contexts(contexts_path, [q["question_id"] for q in answerable[:3]]):
    print("=" * 100)
    print(r["question_id"], f"({r['family']}) detected: {r['engine_type_detected']} | {len(r['hits'])} windows, {r['context_tokens']} tokens, {len(r['reranked'])} reranked")
    print(r["question"])
    for h in r["hits"]:
        print(f"  {h['rank']}. {h['score']:+.3f}  {h['unit_id']}  (p. {h['page_start']}, {h['n_tokens']} tokens)  {h['text'][:110]!r}")

## 9. Checks

1. **Existing run ids are unchanged:** the ids of `grid("v1")` are the stored v1 runs, all complete.
2. **Targets:** overall Hit@1 and evidence in context at or above the targets; Hit@3 not below "X2, top 8". A failure names the families.
3. **No label leakage:** the X3 pipeline holds no labels, its filter is the keyword rule, and `retrieve` receives the question text only.
4. **Cut-off:** every list between `min_keep` and `max_keep`, scores descending, a prefix of `stats["reranked"]`, the cut as long as the rule allows.
5. **Filter:** every candidate has `engine_type ∈ {detected type, all}`; the candidates are the fused top 30 of the cached first-stage lists (unfiltered when no type was detected).
6. **Keyword rule:** the gold sections it removes stay within `MAX_EXCLUDED_GOLD`.
7. **Reproducibility:** a second run from the caches, written to a temporary root, gives identical hits.
8. **Hand-off:** `contexts.jsonl` matches `hits.parquet`, `stats.jsonl` and the corpus.
9. **Metric files:** one row per evaluated run; the baseline rows equal notebook 09's rows in `results/<experiment>/win500/metrics.csv` (when it exists); X3's `section_hit@1` and `section_hit@3` equal the comparison table.

In [ ]:
import inspect

p3 = Pipeline(x3_cfg, res, engine_rules=rules)


def lists_for(text, flt):
    """The cached first-stage lists of X3's two signals for a question and a Qdrant filter."""
    q = p3._query(text)
    return [[(u, s) for u, s in p3.searcher.search(INDEX, sig, q, flt, x3_cfg["candidates"])["hits"]] for sig in x3_cfg["first_stages"]]


grid_ids = {runs.run_id_of(c) for c in grid("v1", prompts=prompts)}
stored_v1 = set(runs.list_runs(params_version="v1"))
with tempfile.TemporaryDirectory() as tmp:
    runs.execute(x3_cfg, Pipeline(x3_cfg, res, engine_rules=rules).run, questions, res.meta(INDEX), root=tmp)
    reproduced = runs.load_hits(X3_ID, tmp).equals(runs.load_hits(X3_ID))
t_csv, t_cmp = table_csv.set_index("run_id"), table.set_index("row")

checks = {
    "1 existing run ids unchanged": (
        [f"in the v1 grid but not stored: {sorted(grid_ids - stored_v1)[:3]}"] * bool(grid_ids - stored_v1)
        + [f"stored v1 runs outside the grid: {sorted(stored_v1 - grid_ids)[:3]}"] * bool(stored_v1 - grid_ids)
        + [f"{r} is not complete" for r in sorted(grid_ids) if not runs.is_complete(r)]),
    "2 targets": final.check_targets(rows, X3_LABEL, "X2, top 8", TARGET_HIT1, TARGET_EVIDENCE),
    "3 no label leakage": (
        ["the X3 pipeline holds labels"] * (p3.labels is not None)
        + ["the X3 filter is not the keyword rule"] * (x3_cfg["filter"]["type"] != "keywords")
        + ["retrieve takes more than the question text"] * (list(inspect.signature(p3.retrieve).parameters) != ["question_id", "text", "k"])),
    "4 cut-off": final.check_cut(X3_ID),
    "5 filter": final.check_filter(X3_ID, questions, corpus, lists_for),
    "6 keyword rule": [f"{len(excluded)} questions lose a gold section (limit {MAX_EXCLUDED_GOLD})"] * (len(excluded) > MAX_EXCLUDED_GOLD),
    "7 reproducibility": ["a second run gives other hits"] * (not reproduced),
    "8 hand-off": final.check_contexts(contexts_path, X3_ID, questions, corpus),
    "9 metric files": (
        final.check_metric_files(OUT, metric_ids, BASELINES, RESULTS, INDEX)
        + ["section_hit@1 differs from the comparison table"] * (abs(t_csv.loc[X3_ID, "section_hit@1"] - t_cmp.loc[X3_LABEL, "hit@1"]) > 1e-9)
        + ["section_hit@3 differs from the comparison table"] * (abs(t_csv.loc[X3_ID, "section_hit@3"] - t_cmp.loc[X3_LABEL, "hit@3"]) > 1e-9)),
}
for name, probs in checks.items():
    print("PASS" if not probs else "FAIL", name, "" if not probs else f"({len(probs)}): {probs[:3]}")
assert not any(checks.values()), "checks failed: see above"
print("all checks passed")

## Next: the generation notebooks

```bash
docker compose stop encoder-gpu && docker compose up -d vllm      # VLLM_CONFIG=serve.retrieval.yaml (.env); ~2.5 min to healthy
docker compose logs -f vllm
```

Read three to five questions from `results/<experiment>/x3/contexts.jsonl` with `final.read_contexts(path, question_ids=[...])` and try other models and prompts on their `hits`. Qdrant and the encoder are not needed.